In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

In [ ]:
# ── 1. Load Data ──────────────────────────────────────────────
cols = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root","num_file_creations",
    "num_shells","num_access_files","num_outbound_cmds","is_host_login",
    "is_guest_login","count","srv_count","serror_rate","srv_serror_rate",
    "rerror_rate","srv_rerror_rate","same_srv_rate","diff_srv_rate",
    "srv_diff_host_rate","dst_host_count","dst_host_srv_count",
    "dst_host_same_srv_rate","dst_host_diff_srv_rate","dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate","dst_host_serror_rate","dst_host_srv_serror_rate",
    "dst_host_rerror_rate","dst_host_srv_rerror_rate","label"
]
df = pd.read_csv("D:/College/College_Notes/Third_Year/Sixth_Sem/MLC-2/Projects/corrected.gz", names=cols)

# ── 2. Preprocessing ──────────────────────────────────────────
df['binary_label'] = df['label'].apply(lambda x: 0 if x == 'normal.' else 1)

for col in ['protocol_type', 'service', 'flag']:
    df[col] = LabelEncoder().fit_transform(df[col])

X = df.drop(['label', 'binary_label'], axis=1)
y = df['binary_label']

X_scaled = StandardScaler().fit_transform(X)

# ── 3. Train/Test Split ───────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# ── 4. Train Random Forest ────────────────────────────────────
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# ── 5. Evaluate ───────────────────────────────────────────────
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Normal", "Anomaly"]))

# ── 6. Confusion Matrix Plot (Matplotlib) ─────────────────────
cm = confusion_matrix(y_test, y_pred)
labels = ["Normal", "Anomaly"]

fig, ax = plt.subplots()
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)

# Add numbers inside the boxes
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i][j]), ha='center', va='center', 
                color='red', fontsize=12)

ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix - Random Forest")
plt.tight_layout()
plt.savefig("confusion_matrix.png")
plt.show()

# ── 7. Feature Importance Plot (Matplotlib) ───────────────────
feature_names = df.drop(['label', 'binary_label'], axis=1).columns
importances = pd.Series(model.feature_importances_, index=feature_names)
top10 = importances.nlargest(10).sort_values(ascending=True)

plt.figure()
plt.barh(top10.index, top10.values, color='steelblue')
plt.title("Top 10 Important Features")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("feature_importance.png")
plt.show()

In [ ]:
# ── 5. Evaluation & Classification Report (Output #1) ──────────
y_pred = model.predict(X_test)
print("1. MODEL PERFORMANCE SUMMARY")
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Normal", "Anomaly"]))

# ── 6. Sample Predictions Table (Output #2) ───────────────────
# This shows a few real-world examples of the model in action
print("\n2. SAMPLE PREDICTIONS (Actual vs. Predicted)")
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
# Mapping back to text for readability
mapping = {0: "Normal", 1: "Anomaly"}
comparison_df['Actual'] = comparison_df['Actual'].map(mapping)
comparison_df['Predicted'] = comparison_df['Predicted'].map(mapping)
print(comparison_df.head(10)) # Shows the first 10 results

# ── 7. Confusion Matrix Plot (Output #3) ──────────────────────
cm = confusion_matrix(y_test, y_pred)
labels = ["Normal", "Anomaly"]
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(labels); ax.set_yticklabels(labels)

# Add text to the boxes
thresh = cm.max() / 2.
for i in range(2):
    for j in range(2):
        ax.text(j, i, format(cm[i, j], 'd'), ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")

ax.set_title("3. Confusion Matrix")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.show()

# ── 8. Feature Importance Plot (Output #4) ────────────────────
feature_names = X.columns
importances = pd.Series(model.feature_importances_, index=feature_names)
top10 = importances.nlargest(10).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top10.index, top10.values, color='steelblue')
plt.title("4. Top 10 Most Important Network Features")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()